In [ ]:
import pandas as pd
from pathlib import Path

input_file_agg = "aggregated_results.csv"

df_agg = pd.read_csv(input_file_agg)
df_agg.head()

In [ ]:
# create targets mapping[subject][issue/BIC/BFC]
from collections import defaultdict
targets = defaultdict(dict)
for index, row in df_agg.iterrows():
    subject = row['subject']
    issue = row['iid']
    commit = row['commit']
    targets[subject]['issue'] = issue
    if row['scenario'] == 'FIX':
      targets[subject]['BFC'] = commit
    else:
      targets[subject]['BIC'] = commit

print(targets)


In [ ]:
from difflib import SequenceMatcher
from termcolor import colored

def word_diff(a, b, latex=False):
  def preprocess(text):
    return text.replace('.', '').replace(',', '')

  matcher = SequenceMatcher(None, preprocess(a).split(), preprocess(b).split())
  diff = []
  for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == 'replace':
      if latex:
        diff.append(f"\\textcolor{{red}}{{{' '.join(a.split()[i1:i2])}}}")
        diff.append(f"\\textcolor{{teal}}{{{' '.join(b.split()[j1:j2])}}}")
      else:
        diff.append(colored(' '.join(a.split()[i1:i2]), 'red'))
        diff.append(colored(' '.join(b.split()[j1:j2]), 'green'))
    elif tag == 'delete':
      if latex:
        diff.append(f"\\textcolor{{red}}{{{' '.join(a.split()[i1:i2])}}}")
      else:
        diff.append(colored(' '.join(a.split()[i1:i2]), 'red'))
    elif tag == 'insert':
      if latex:
        diff.append(f"\\textcolor{{teal}}{{{' '.join(b.split()[j1:j2])}}}")
      else:
        diff.append(colored(' '.join(b.split()[j1:j2]), 'green'))
    elif tag == 'equal':
      diff.append(' '.join(a.split()[i1:i2]))
  return ' '.join(diff)

# Example usage
msg_origin = 'Fix parsing of invalid asterix symbol after private field symbol.'
msg_enhanced = 'Fix parsing of invalid asterisk symbol after private field symbol, which previously caused a segmentation fault when an asterisk was used after an async private method declaration.'

# Output in terminal with colors
print(word_diff(msg_origin, msg_enhanced))

# Output in LaTeX format
print(word_diff(msg_origin, msg_enhanced, latex=True))

In [ ]:
import yaml

msgyaml = '../msg/gemini.yaml'
with open(msgyaml, 'r') as f:
    msgdata = yaml.safe_load(f)

enhance_score_improve = []
enhance_result_improve = []
enhance2B = []
enhance_N2B = []
enhance_R2B = []
enhance_D2B = []

reduce_score_worse = []
reduce_result_worse = []

for item in msgdata:
  proj, issue, commit = item['proj'], item['issue'], item['commit']
  score_msgonly = item['score_msgonly']
  score_enhance = item.get('score_enhanced', None)
  score_reduce = item.get('score_reduced', None)
  result_msgonly = item['best_result_msgonly']
  result_enhance = item.get('best_result_enhanced', None)
  result_reduce = item.get('best_result_reduced', None)
  if score_enhance is not None and score_enhance > score_msgonly:
    enhance_score_improve.append(commit)
  if result_msgonly != 'B' and result_enhance == 'B':
    enhance2B.append(commit)
  if result_msgonly == 'N' and result_enhance == 'B':
    enhance_N2B.append(commit)
  if result_msgonly == 'R' and result_enhance == 'B':
    enhance_R2B.append(commit)
  if result_msgonly == 'D' and result_enhance == 'B':
    enhance_D2B.append(commit)
  
  if score_reduce is not None and score_reduce < score_msgonly:
    reduce_score_worse.append(commit)

print(f"Enhance improve score: {len(enhance_score_improve)}")
print(f"Enhance 2B: {len(enhance2B)}")
print(f"Enhance N2B: {len(enhance_N2B)}")
print(f"Enhance R2B: {len(enhance_R2B)}")
print(f"Enhance D2B: {len(enhance_D2B)}")
print(f"Reduce score worse: {len(reduce_score_worse)}")


In [ ]:
import re
# generate table for reduce score worse with following header
# proj, issue, msg, msg_reduce, score_msgonly, score_reduce, result_msgonly, result_reduce

table_reduce_worse = []
for item in msgdata:
  proj, issue, commit = item['proj'], item['issue'], item['commit']
  score_msgonly = item['score_msgonly']
  score_reduce = item.get('score_reduced', None)
  result_msgonly = item['best_result_msgonly']
  result_reduce = item.get('best_result_reduced', None)
  msg = item['msg']
  msg_reduce = item.get('msg_reduced', None)
  # calculate reduced words
  if msg_reduce:
    words_msg = re.findall(r'\S+', msg)
    tokens_reduce = re.findall(r'\S+', msg_reduce)
    wc_reduced = len(words_msg) - len(tokens_reduce)
  if commit in reduce_score_worse:
    table_reduce_worse.append({
      'proj': proj,
      'issue': issue,
      'msg': msg,
      'msg_reduce': msg_reduce,
      'score_msgonly': score_msgonly,
      'score_reduce': score_reduce,
      'result_msgonly': result_msgonly,
      'result_reduce': result_reduce,
      'wc_reduced': wc_reduced,
    })

df_table_reduce_worse = pd.DataFrame(table_reduce_worse)
display(df_table_reduce_worse)
# acount avg reduced words
avg_wc_reduced = df_table_reduce_worse['wc_reduced'].mean()
print(f"Average reduced words: {avg_wc_reduced}")

In [ ]:
def openai_count_token(s: str) -> int:
  import tiktoken
  encoding = tiktoken.encoding_for_model("gpt-4o")
  tokens = encoding.encode(s)
  return len(tokens)

# generate table for enhance2B worse with following headr
# proj, issue, msg, msg_enhance, score_msgonly, score_enhance, result_msgonly, result_enhance
table_enhance_2B = []
for item in msgdata:
  proj, issue, commit = item['proj'], item['issue'], item['commit']
  score_msgonly = item['score_msgonly']
  score_enhance = item.get('score_enhanced', None)
  result_msgonly = item['best_result_msgonly']
  result_enhance = item.get('best_result_enhanced', None)
  msg = item['msg']
  msg_enhance = item.get('msg_enhanced', None)
  if msg_enhance:
    words_msg = re.findall(r'\S+', msg)
    words_enhance = re.findall(r'\S+', msg_enhance)
    # wc_added = len(tokens_enhance) - len(tokens_msg)
    wc_added = len(words_enhance) - len(words_msg)
  else:
    wc_added = 0
  if commit in enhance2B:
    table_enhance_2B.append({
      'proj': proj,
      'issue': issue,
      'msg': msg,
      'msg_enhance': msg_enhance,
      'score_msgonly': score_msgonly,
      'score_enhance': score_enhance,
      'result_msgonly': result_msgonly,
      'result_enhance': result_enhance,
      'wc_added': wc_added,
    })
df_table_enhance_2B = pd.DataFrame(table_enhance_2B)
display(df_table_enhance_2B)

In [ ]:
def openai_count_token(s: str) -> int:
  import tiktoken
  encoding = tiktoken.encoding_for_model("gpt-4o-2024-08-06")
  tokens = encoding.encode(s)
  return len(tokens)

# generate table for enhance2B worse with following headr
# proj, issue, msg, msg_enhance, score_msgonly, score_enhance, result_msgonly, result_enhance
table_msg = []
for item in msgdata:
  proj, issue, commit = item['proj'], item['issue'], item['commit']
  score_msgonly = item['score_msgonly']
  score_enhance = item.get('score_enhanced', None)
  score_reduce = item.get('score_reduced', None)
  result_msgonly = item['best_result_msgonly']
  result_enhance = item.get('best_result_enhanced', None)
  result_reduce = item.get('best_result_reduced', None)
  msg = item['msg']
  msg_enhance = item.get('msg_enhanced', None)
  msg_reduce = item.get('msg_reduced', None)
  wc_added = wc_reduced = 0
  if msg_enhance:
    # wc_added = openai_count_token(msg_enhance) - openai_count_token(msg)
    words_msg = re.findall(r'\S+', msg)
    words_enhance = re.findall(r'\S+', msg_enhance)
    wc_added = len(words_enhance) - len(words_msg)
  if msg_reduce:
    # wc_reduced = openai_count_token(msg) - openai_count_token(msg_reduce)
    words_msg = re.findall(r'\S+', msg)
    words_reduce = re.findall(r'\S+', msg_reduce)
    wc_reduced = len(words_msg) - len(words_reduce)
  table_msg.append({
    'proj': proj,
    'issue': issue,
    'msg': msg,
    'msg_enhance': msg_enhance,
    'msg_reduce': msg_reduce,
    'score_msgonly': score_msgonly,
    'score_enhance': score_enhance,
    'score_reduce': score_reduce,
    'result_msgonly': result_msgonly,
    'result_enhance': result_enhance,
    'result_reduce': result_reduce,
    'wc_added': wc_added,
    'wc_reduced': wc_reduced,
  })
df_table_msg = pd.DataFrame(table_msg)
display(df_table_msg)

# show wc_added avg, only for rows with msg_enhance not None
df_table_msg = df_table_msg[df_table_msg['msg_enhance'].notna()]
avg_wc_added = df_table_msg['wc_added'].mean()
# calculate percentage of added words over original msg
print(f"Average added words: {avg_wc_added}")
print(df_table_msg['wc_added'].describe())

In [ ]:
enhanced_examples = [7252, 65, 5153]

for issue in enhanced_examples:
  msg = df_table_enhance_2B[df_table_enhance_2B['issue'] == issue]['msg'].values[0]
  msg = msg.splitlines()[0]
  msg_enhance = df_table_enhance_2B[df_table_enhance_2B['issue'] == issue]['msg_enhance'].values[0]
  msg_enhance = msg_enhance.splitlines()[0]
  msg_diff = word_diff(msg, msg_enhance, latex=True)
  print(f"Issue {issue} diff:")
  print(msg_diff)

reduced_examples = [5117]
for issue in reduced_examples:
  msg = df_table_reduce_worse[df_table_reduce_worse['issue'] == issue]['msg'].values[0]
  msg = msg.splitlines()[0]
  msg_reduce = df_table_reduce_worse[df_table_reduce_worse['issue'] == issue]['msg_reduce'].values[0]
  msg_reduce = msg_reduce.splitlines()[0]
  msg_diff = word_diff(msg, msg_reduce, latex=True)
  print(f"Issue {issue} diff:")
  print(msg_diff)
